In [61]:
import logging

logging.basicConfig(level=logging.WARNING)

from mascaf import *
from swctools import SWCModel, FrustaSet, PointSet, plot_model
import numpy as np

import os

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [72]:
spine_idx = 1
mcf_qst = 0.5
mcf_mcst = 5
obj_name = f"TS{spine_idx}"
polylines_name = f"TS{spine_idx}_qst{mcf_qst}_mcst{mcf_mcst}"
mm = MeshManager(mesh_path=f"../data/mesh/processed/{obj_name}.obj")
raw_skeleton = SkeletonGraph.from_txt(
    f"../data/mcf_skeletons/{polylines_name}.polylines.txt"
)
raw_skeleton.prune_short_branches_inplace(min_length_percentile=20)

# mesh only
fig = mm.visualize_mesh_3d(skel=None, show_axes=False, title="")
eye_coord = np.array([0.5, 2.0, 1.0]) / 1.8
fig.update_layout(
    scene=dict(
        camera=dict(
            eye={'x':eye_coord[0], 'y':eye_coord[1], 'z':eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # or 'cube', 'auto', 'manual'
    )
)
fig.show()

# mesh with skeleton
fig = mm.visualize_mesh_3d(skel=raw_skeleton, show_axes=False, title="")
fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # or 'cube', 'auto', 'manual'
    )
)
fig.show()

In [68]:
opts = SkeletonOptimizerOptions(
    max_iterations=20,
    step_size=2.0,
    smoothing_weight=0.1,
    preserve_terminal_nodes=True,
    preserve_branch_nodes=False,
    verbose=True,
)
optimizer = SkeletonOptimizer(raw_skeleton, mm.mesh, opts)
optimized_skeleton = optimizer.optimize()
skeleton = optimized_skeleton
fig = mm.visualize_mesh_3d(
    skel=[raw_skeleton, optimized_skeleton],
    skel_color=["crimson", "blue"],
    title="",
    show_axes=False,
)
fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # or 'cube', 'auto', 'manual'
    )
)
fig.show()

In [69]:
max_edge_length = 200
radius_strategy = "equivalent_area"

swc_out_dir = f"../data/swc/current/{polylines_name}"
swc_filepath = f"{swc_out_dir}/TS{spine_idx}_s{max_edge_length}_{radius_strategy}.swc"

# check if directory exists, if not create it
if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

morph = fit_morphology(
    mm.mesh,
    skeleton,
    options=FitOptions(
        max_edge_length=max_edge_length,
        radius_strategy=radius_strategy,
        snap_polylines_to_mesh=True,
    ),
)
# write swc to file
morph.to_swc_file(swc_filepath)
# validation
# validator = Validation(mm, skeleton, morph)
# validator.full_validation()

model = SWCModel.from_swc_file(swc_filepath)
model.print_attributes(node_info=False, edge_info=False)
frusta = FrustaSet.from_swc_model(model)
title = f"TS{spine_idx}_s{max_edge_length}_{radius_strategy}"
fig = plot_model(
    swc_model=model,
    frusta=frusta,
    slider=False,
    title="",
    width=800,
    height=600,
    hide_axes=True,
)
fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # or 'cube', 'auto', 'manual'
    )
)
fig.show()

SWCModel: nodes=89, edges=88, components=1, cycles=0, branch_points=12, roots=1, leaves=16, self_loops=0, density=0.0225


In [70]:
morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)

# save normalized to file
swc_filepath = (
    f"{swc_out_dir}/TS{spine_idx}_s{max_edge_length}_{radius_strategy}_SAnorm.swc"
)
morph.to_swc_file(swc_filepath)

# load and plot
swc_model = SWCModel.from_swc_file(swc_filepath)
swc_model.print_attributes(node_info=False, edge_info=False)
frusta = FrustaSet.from_swc_model(swc_model)
title = f"TS{spine_idx}_s{max_edge_length}_{radius_strategy}"
fig = plot_model(
    swc_model=swc_model,
    frusta=frusta,
    slider=False,
    title="",
    width=800,
    height=600,
    hide_axes=True,
)
fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # or 'cube', 'auto', 'manual'
    )
)
fig.show()

# validator = Validation(mm, skeleton, morph)
# validator.full_validation()

SWCModel: nodes=89, edges=88, components=1, cycles=0, branch_points=12, roots=1, leaves=16, self_loops=0, density=0.0225
